<a href="https://colab.research.google.com/github/ishaanrai-hub/IIT-Hyd-Projects-Machine-learning-/blob/main/Telco_Customer_Churn_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
# Question 17: Multi-Classifier System Design and Performance Analysis
# Scenario:

# You are building a customer churn prediction system for a telecommunications company. The system must predict whether a customer will leave the company within the next 30 days.

# Dataset: Telco Customer Churn Dataset

# Link: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

# Dataset Characteristics:

# 7,043 customer records
# 20 features (mix of numerical and categorical)
# Target: Churn (Yes/No)
# Class distribution: ~73% No Churn, ~27% Churn
# Business Requirements:

# Identifying 70% of churners is minimum acceptable (high recall priority)
# Customer retention team can only contact 2000 customers per month
# Need to explain to executives why customers are predicted to churn
# System must update predictions daily as new data arrives
# Cost of retention offer: $50 per customer
# Average revenue lost from churn: $500 per customer
# Your Task:

# Design and implement a complete machine learning solution addressing:

# 1. Data Preprocessing Pipeline (20%):

# Design a comprehensive preprocessing pipeline handling missing values, scaling, and encoding
# Justify each preprocessing choice for each classifier type
# Explain how to handle the pipeline for daily updates
# Provide complete implementation code
# 2. Multi-Classifier Implementation and Comparison (30%):

# Implement all four classifiers (Logistic Regression, k-NN, SVM, Decision Tree)
# Configure each classifier optimally for the business problem
# Use appropriate cross-validation strategy
# Create a detailed comparison table with all relevant metrics
# 3. Performance Analysis and Business Impact (30%):

# Analyze results using appropriate metrics for imbalanced data
# Calculate business value/cost for each classifier
# Create visualizations comparing classifier performance
# Recommend optimal classification threshold based on business constraints
# Calculate ROI for each model
# Justify which classifier(s) to deploy
# 4. Production Deployment Strategy (20%):

# Design monitoring system to track model performance over time
# Explain how to handle concept drift
# Propose A/B testing strategy
# Describe retraining pipeline and decision criteria
# Evaluation Rubric:

# Data preprocessing: 4 points
# Multi-classifier implementation: 6 points
# Performance analysis and business impact: 6 points
# Production deployment strategy: 4 points

In [58]:
import pandas as pd
import numpy as np

# Load dataset (ensure CSV is in working directory)
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [59]:
df.info()
df['Churn'].value_counts(normalize=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,proportion
Churn,
No,0.73463
Yes,0.26537


In [60]:
# ## 2. Preprocessing Pipeline

# We use:
# - Median imputation for numeric features
# - Most-frequent imputation for categorical features
# - StandardScaler for distance-based models (k-NN, SVM, Logistic Regression)
# - OneHotEncoding for categorical variables

# Pipelines allow **daily updates** by reusing fitted transformers.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

In [61]:
## 3. Multi-Classifier Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'k-NN': KNeighborsClassifier(n_neighbors=15),
    'SVM': SVC(kernel='rbf', probability=True, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced')
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['precision', 'recall', 'f1', 'roc_auc']

results = {}
for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    results[name] = {metric: scores[f'test_{metric}'].mean() for metric in scoring}

pd.DataFrame(results).T

,precision,recall,f1,roc_auc
Logistic Regression,0.559659,0.733779,0.634804,0.845486
k-NN,0.592819,0.563880,0.577807,0.820078
SVM,0.515070,0.786622,0.622270,0.835978
Decision Tree,0.504744,0.788629,0.615291,0.818749


In [62]:
## 4. Performance & Business Impact Analysis
from sklearn.metrics import precision_recall_curve

# Fit best model (Logistic Regression for explainability)
best_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', models['Logistic Regression'])
])
best_pipe.fit(X_train, y_train)

probs = best_pipe.predict_proba(X_test)[:,1]
precision, recall, thresholds = precision_recall_curve(y_test, probs)

# Business constants
CONTACT_LIMIT = 2000
RETENTION_COST = 50
CHURN_LOSS = 500

df_eval = pd.DataFrame({
    'threshold': np.append(thresholds, 1),
    'precision': precision,
    'recall': recall
})

# Filter thresholds meeting recall constraint
df_eval = df_eval[df_eval['recall'] >= 0.7]
df_eval.head()

,threshold,precision,recall
0,0.006066,0.265436,1.0
1,0.006227,0.265625,1.0
2,0.006244,0.265814,1.0
3,0.006254,0.266003,1.0
4,0.006468,0.266192,1.0


In [63]:
# Select threshold under contact constraint
df_eval['contacts'] = (probs[:,None] >= df_eval['threshold'].values).sum(axis=0)
df_eval = df_eval[df_eval['contacts'] <= CONTACT_LIMIT]
df_eval.sort_values(by='precision', ascending=False).head()

,threshold,precision,recall,contacts
926,0.522748,0.546778,0.703209,481
927,0.523522,0.545833,0.700535,480
925,0.521176,0.545643,0.703209,482
924,0.520483,0.544513,0.703209,483
918,0.518533,0.543967,0.711230,489


In [64]:
## 5. ROI Calculation
tp = int(0.7 * y_test.sum())
cost = CONTACT_LIMIT * RETENTION_COST
saved = tp * CHURN_LOSS
roi = (saved - cost) / cost

roi

0.305

In [65]:
# ## 6. Deployment & Monitoring Strategy

# - **Monitoring**: Track recall, precision, data drift (PSI)
# - **Concept Drift**: Retrain if recall < 65% or PSI > 0.2
# - **A/B Testing**: Champion/Challenger with monthly evaluation
# - **Retraining**: Weekly batch retraining with rolling window

## Final Recommendation

# **Deploy Logistic Regression** as primary model:
# - Meets recall requirement
# - High interpretability for executives
# - Stable ROI

# **Decision Tree** as challenger model for non-linear patterns.
